# kuku-promos: find promo clips in microdrama screenplays

The LLM picks moments by **beat index**; code resolves timestamps, snaps cuts so no dialogue is chopped, repairs duration, and verifies every quoted line against the source.

**Confidentiality:** you upload the screenplay pack yourself in step 2. Nothing is hosted with this notebook. Outputs contain screenplay text, so do not publish them.

In [ ]:
# 1. Get the code (set REPO_URL to the code-only repo) and install
REPO_URL = "https://github.com/ShreyanshMehra/kuku-promos"
!git clone $REPO_URL kuku-promos
%cd kuku-promos
!pip -q install -r requirements.txt

In [ ]:
# 2. Upload the screenplay pack (.zip, or several .txt files)
import os
from google.colab import files
os.makedirs("input", exist_ok=True)
for name, data in files.upload().items():
    open(os.path.join("input", name), "wb").write(data)
print(os.listdir("input"))

In [ ]:
# 3. Choose the model. Everything is environment variables -- no code edits.
import os, getpass
USE_MOCK = False  # True = offline placeholder (fixed last-~38s window, NOT a model judgement)
if USE_MOCK:
    os.environ["LLM_PROVIDER"] = "mock"
else:
    os.environ["LLM_PROVIDER"] = "gemini"   # or "openai_compatible" with OPENAI_API_KEY / OPENAI_BASE_URL
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Gemini API key (free: aistudio.google.com/apikey): ")
    # os.environ["LLM_MODEL"] = "gemini-3.6-flash"
    # os.environ["JUDGE_MODEL"] = "gemini-3.5-flash-lite"

In [ ]:
# 4. Run. Re-running is free: responses are cached on disk.
import glob
src = (glob.glob("input/*.zip") or ["input"])[0]
!python run.py --input "$src" --out out

In [ ]:
# 5. Look at the ranked shortlist and the quality report
import pandas as pd
from IPython.display import Markdown, display
df = pd.read_csv("out/promos.csv")
display(df[["rank", "feed", "episode", "start", "end", "duration_s", "hook_line", "cliffhanger_line", "why_it_works", "final_score"]].head(15))
display(Markdown(open("out/metrics.md", encoding="utf-8").read()))

In [ ]:
# 6. Download everything (contains screenplay text; share privately)
!zip -qr out.zip out
files.download("out.zip")